## Prompt Canonicalization Experiments

This notebook documents a series of controlled A/B experiments designed to determine the most reliable structural configuration for zero-shot LilyPond generation.
Each test isolates one prompt parameter, such as syntax style, container structure, or use of functions, and evaluates 20 independent generations using identical model settings.

For every configuration, we measure:

Validity (whether the LilyPond code compiles successfully)

Constraint adherence (compliance with bar count, rhythm, and syntax rules)

Runtime and token usage (efficiency indicators)

Diversity metrics (interval entropy, direction changes, step share, etc.)

The goal of these experiments is to empirically identify a canonical LilyPond prompt format that maximizes syntactic correctness, stability, and reproducibility, while minimizing ambiguity and token overhead.
All subsequent zero-shot and fine-tuning experiments in this project will adopt the configuration established here as the baseline generation template.

## Code Overview

The following code executes the automated prompt comparison pipeline.
For each tested configuration, it sends the corresponding instruction prompt to the model, collects the raw responses, and evaluates them on syntax validity, constraint adherence, token usage, runtime, and musical diversity metrics.
All results are aggregated into structured summaries and stored for analysis and visualization in later sections.

In [ ]:
from openai import OpenAI
from IPython.display import display, Markdown
from pathlib import Path
import os
import re
import shutil
import subprocess
import tempfile
from datetime import datetime
import math
from collections import Counter

# =========================
# LilyPond discovery
# =========================
def find_lilypond():
    r"""Try to find LilyPond executable."""
    # PATH
    lp = shutil.which("lilypond")
    if lp:
        try:
            if subprocess.run([lp, "--version"], capture_output=True, timeout=5).returncode == 0:
                return lp
        except:
            pass
    # Environment variable
    env_path = os.environ.get("LILYPOND_BIN")
    if env_path and Path(env_path).exists():
        try:
            if subprocess.run([env_path, "--version"], capture_output=True, timeout=5).returncode == 0:
                return env_path
        except:
            pass
    # Common Windows install path
    default_path = r"C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe"
    if Path(default_path).exists():
        try:
            if subprocess.run([default_path, "--version"], capture_output=True, timeout=5).returncode == 0:
                return default_path
        except:
            pass
    return None

# =========================
# Extraction & validation
# =========================
def extract_lilypond_code(text: str) -> str:
    r"""Extract LilyPond code from text, removing markdown code blocks if present."""
    text = re.sub(r'```(?:lilypond)?\s*\n?(.*?)```', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^(?:Here|Here\'s|Here is|Output|LilyPond code:|The following).*?:\s*', '', text, flags=re.IGNORECASE)
    return text.strip()

def validate_lilypond(code: str, lilypond_path: str):
    r"""Validate if LilyPond code compiles. Returns (is_valid, error_message)."""
    if not code.strip():
        return False, "Empty code"
    if '\\version' not in code:
        code = '\\version "2.24.4"\n' + code
    with tempfile.TemporaryDirectory() as tdir:
        tmp_file = Path(tdir) / "test.ly"
        tmp_file.write_text(code, encoding="utf-8")
        try:
            result = subprocess.run(
                [lilypond_path, "-dno-print-pages", "-dbackend=null",
                 "-o", str(Path(tdir) / "test"), str(tmp_file)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=10,
                encoding="utf-8"
            )
            if result.returncode == 0:
                return True, ""
            else:
                error = (result.stderr or result.stdout or "Unknown error").strip()
                # Scrub absolute paths for readability
                error = re.sub(r'[A-Za-z]:[\\/][^\s:]+[\\/]([^:]+):', r'\1:', error)  # Windows
                error = re.sub(r'/[^\s:]+/([^:]+):', r'\1:', error)                  # Unix
                error = re.sub(r'[A-Za-z]:[\\/][^\s:]+[\\/]', '', error)
                error = re.sub(r'/[^\s:]+/', '', error)
                lines_err = [l for l in error.split('\n') if 'error' in l.lower()]
                if lines_err:
                    return False, lines_err[0][:200]
                lines = [l.strip() for l in error.split('\n') if l.strip()]
                return False, lines[0][:200] if lines else "Compilation failed"
        except subprocess.TimeoutExpired:
            return False, "Timeout"
        except Exception as e:
            return False, str(e)[:150]

# =========================
# Lightweight melody parser + metrics
# =========================
NOTE2SEMITONE = dict(zip("c d e f g a b".split(), [0,2,4,5,7,9,11]))

def _extract_body(text: str) -> str:
    code = re.sub(r'%.*', '', text)  # strip comments
    m = re.search(r'\{(.*)\}', code, re.S)
    return m.group(1) if m else code

def _tokenize_music(text):
    """Tokenize LilyPond music body into note tokens and bar separators."""
    body = _extract_body(text)
    # Return note tokens like a4, b'4, c, and the bar separator '|'
    return re.findall(r"[a-g][\',]*4|\|", body)

def _midi_from_abs(name, marks, base_oct=3):
    octave = base_oct + marks.count("'") - marks.count(",")
    return 12 * (octave + 1) + NOTE2SEMITONE[name]

def _resolve_relative(names_simple, start_midi=60):
    resolved = []
    prev = start_midi
    for name in names_simple:
        pc = NOTE2SEMITONE[name]
        base_oct = prev // 12
        candidates = [12 * (base_oct + k) + pc for k in (-2,-1,0,1,2)]
        midi = min(candidates, key=lambda x: abs(x - prev))
        resolved.append(midi)
        prev = midi
    return resolved

def _resolve_mixed(names, marks, *, start_midi=60, base_oct=3):
    midis = []
    prev = start_midi
    for n, mk in zip(names, marks):
        if mk:
            midi = _midi_from_abs(n, mk, base_oct=base_oct)
        else:
            pc = NOTE2SEMITONE[n]
            base = prev // 12
            candidates = [12 * (base + k) + pc for k in (-2,-1,0,1,2)]
            midi = min(candidates, key=lambda x: abs(x - prev))
        midis.append(midi)
        prev = midi
    return midis

def _split_bars(seq):
    bars, cur = [], []
    for x in seq:
        if x == "|":
            if cur:
                bars.append(cur)
                cur = []
        else:
            cur.append(x)
    if cur:
        bars.append(cur)
    return bars

def _intervals(pitches):
    return [pitches[i+1] - pitches[i] for i in range(len(pitches)-1)]

def _entropy(values):
    if not values:
        return 0.0
    c = Counter(values); n = sum(c.values())
    return -sum((v/n)*math.log2(v/n) for v in c.values())

def _in_key_c_major(midi):
    return (midi % 12) in {0,2,4,5,7,9,11}

def analyze_lilypond(text, *, enforce_range_for_relative=False, range_bounds=(60,72), expected_notation="relative"):
    """
    Returns both constraint adherence flags and diversity metrics.
    Works for absolute and relative (heuristically resolves relative).
    """
    # String-level quick checks
    rel_count = len(re.findall(r"\\relative\b", text))
    uses_relative = rel_count > 0

    # Robust anchor diagnostics (not required for pass)
    has_rel_cprime = bool(re.search(r"\\relative\s+c'\s*\{", text))
    has_rel_c_plain = bool(re.search(r"\\relative\s+c\s*\{", text))
    # Main constraint: anchor must be \relative c' exactly once

    has_music_block = bool(re.search(r"\{.*\}", text, re.S))
    has_key_time = ("\\key c \\major" in text) and ("\\time 4/4" in text)
    lowercase_ok = not re.search(r"[A-G]", text)

    # Body for body-only checks
    body = _extract_body(text)
    has_octave_marks = bool(re.search(r"\b[a-g][',]+\d\b", body))
    notation_mode = (expected_notation or "relative").lower()
    if notation_mode not in {"relative", "absolute"}:
        notation_mode = "relative"
    if notation_mode == "relative":
        notation_ok = (rel_count == 1 and has_rel_cprime and not has_octave_marks)
    else:
        notation_ok = (rel_count == 0 and has_octave_marks)

    # Durations: only '4' allowed (inside body only)
    durations_ok = not re.search(r"(?<!\d)(?:1|2|8|16|32|64)(?!\d)", body)

    # ---- Safer forbidden detection ----
    rest_forbidden = re.search(r"(?<!\\)\br\b", body) is not None
    chord_forbidden = ("<" in body) or (">" in body)
    voices_forbidden = re.search(r"<<|\\\\|>>", text) is not None
    repeat_forbidden = re.search(r"\\repeat\b", text) is not None
    score_forbidden  = re.search(r"\\score\b",  text) is not None
    layout_forbidden = re.search(r"\\layout\b", text) is not None
    forbidden = any([rest_forbidden, chord_forbidden, voices_forbidden,
                     repeat_forbidden, score_forbidden, layout_forbidden])

    tokens = _tokenize_music(text)
    if not tokens:
        rel_ok = (has_rel_cprime and rel_count == 1)
        return {
            "uses_relative": uses_relative,
            "relative_ok": rel_ok,
            "relative_anchor_cprime": has_rel_cprime,
            "relative_anchor_plain_c": has_rel_c_plain,
            "notation_ok": notation_ok,
            "has_music_block": has_music_block,
            "bars_ok": False,
            "notes_per_bar_ok": False,
            "range_ok": False,
            "has_key_time": has_key_time,
            "durations_ok": durations_ok,
            "lowercase_ok": lowercase_ok,
            "no_forbidden": not forbidden,
            "adherence": 0.0,
            "interval_entropy": 0.0,
            "repeat_rate": 1.0,
            "step_vs_leap": 0.0,
            "direction_changes": 0,
            "key_conform_pct": 0.0,
            "ends_on_tonic": 0
        }

    # Parse note tokens
    bar_split = _split_bars(tokens)
    bars_ok = (len(bar_split) == 8)
    notes_per_bar_ok = all(len(b) == 4 for b in bar_split) if bar_split else False

    # Extract note names + octave marks
    names, marks = [], []
    for t in tokens:
        if t == "|":
            continue
        m = re.fullmatch(r"([a-g])([',]*)4", t)
        if not m:
            continue
        n, mk = m.groups()
        names.append(n); marks.append(mk)

    # Decide absolute vs relative and get MIDI (handles mixed correctly)
    any_marks = any(bool(mk) for mk in marks)
    all_marks = all(bool(mk) for mk in marks)
    if not any_marks:
        midis = _resolve_relative(names, start_midi=60)
    elif all_marks:
        midis = [_midi_from_abs(n, mk, base_oct=3) for n, mk in zip(names, marks)]
    else:
        midis = _resolve_mixed(names, marks, start_midi=60, base_oct=3)

    # Range policy
    lo, hi = range_bounds
    if any_marks or enforce_range_for_relative:
        rng_ok = bool(midis) and (min(midis) >= lo and max(midis) <= hi)
    else:
        rng_ok = True

    # Diversity metrics
    intrv = _intervals(midis)
    interval_entropy = _entropy(intrv)
    repeat_rate = (sum(1 for i in range(1, len(midis)) if midis[i] == midis[i-1]) /
                   max(1, len(midis)-1))
    steps = sum(1 for d in intrv if abs(d) <= 2)
    leaps = sum(1 for d in intrv if abs(d) > 2)
    total_motions = steps + leaps
    step_vs_leap = steps / total_motions if total_motions else 0.0
    direction_changes = sum(
        1 for i in range(1, len(intrv))
        if (intrv[i-1] < 0 and intrv[i] > 0) or (intrv[i-1] > 0 and intrv[i] < 0)
    )
    key_conform_pct = (sum(_in_key_c_major(n) for n in midis) / len(midis)) if midis else 0.0
    ends_on_tonic = 1 if (midis and (midis[-1] % 12) == 0) else 0

    # Relative correctness (main)
    rel_ok = (has_rel_cprime and rel_count == 1)

    # Adherence flags
    flags = [
        has_music_block,
        has_key_time, bars_ok, notes_per_bar_ok,
        durations_ok, lowercase_ok, (not forbidden), rng_ok, notation_ok
    ]
    adherence = sum(flags) / len(flags)

    return {
        "uses_relative": uses_relative,
        "relative_ok": rel_ok,
        "relative_anchor_cprime": has_rel_cprime,
        "relative_anchor_plain_c": has_rel_c_plain,
        "notation_ok": notation_ok,
        "has_music_block": has_music_block,
        "bars_ok": bars_ok,
        "notes_per_bar_ok": notes_per_bar_ok,
        "durations_ok": durations_ok,
        "lowercase_ok": lowercase_ok,
        "no_forbidden": not forbidden,
        "has_key_time": has_key_time,
        "range_ok": rng_ok,
        "adherence": adherence,
        "interval_entropy": interval_entropy,
        "repeat_rate": repeat_rate,
        "step_vs_leap": step_vs_leap,
        "direction_changes": direction_changes,
        "key_conform_pct": key_conform_pct,
        "ends_on_tonic": ends_on_tonic
    }

# =========================
# Main runner
# =========================
def run_zero_shot_test(
    prompt_file: str | Path = "../configs/prompts/zero_shot.txt",
    model: str = "openai/gpt-oss-20b:fireworks-ai",
    num_runs: int = 20,
    max_tokens: int = 10000,
    temperature: float = 0.7,
    *,
    expected_notation="relative",
    enforce_range_for_relative=False,
    range_bounds=(60,72)
):
    prompt_file = Path(prompt_file)

    lilypond_path = find_lilypond()
    if lilypond_path:
        display(Markdown(f"✅ **LilyPond found:** `{lilypond_path}`\n"))
    else:
        display(Markdown("⚠️ **LilyPond not found** — compilation validation will be skipped.\n"))

    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        display(Markdown("**❌ Error:** `HF_TOKEN` environment variable not set!"))
        return

    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    if not prompt_file.exists():
        display(Markdown(f"**❌ Error:** Prompt file not found: `{prompt_file}`"))
        return
    prompt_text = prompt_file.read_text(encoding="utf-8").strip()
    display(Markdown(f"### 📝 Input Prompt\n```\n{prompt_text}\n```\n\n---\n"))

    results = []
    total_start_time = datetime.now()

    for run_num in range(1, num_runs + 1):
        display(Markdown(f"## 🔄 Run {run_num}/{num_runs}\n"))
        start_time = datetime.now()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        elapsed = (datetime.now() - start_time).total_seconds()

        choice = resp.choices[0]
        raw_text = choice.message.content if choice.message and choice.message.content else ""
        lilypond_code = extract_lilypond_code(raw_text) if raw_text else ""

        is_valid, validation_error = None, None
        if lilypond_code and lilypond_path:
            is_valid, validation_error = validate_lilypond(lilypond_code, lilypond_path)

        metrics = analyze_lilypond(
            lilypond_code,
            enforce_range_for_relative=enforce_range_for_relative,
            range_bounds=range_bounds,
            expected_notation=expected_notation
        ) if lilypond_code else {}

        # --- soft quality gates (analysis-only) ---
        if metrics:
            metrics["entropy_ok"] = metrics.get("interval_entropy", 0) >= 1.20
            metrics["contour_ok"] = metrics.get("direction_changes", 0) >= 2
            metrics["variety_ok"] = metrics.get("step_vs_leap", 1.0) <= 0.95  # ≥5% leaps
            if metrics["step_vs_leap"] > 0.95 or metrics["interval_entropy"] < 1.20:
                metrics["style"] = "scalar"
            elif metrics["direction_changes"] >= 2:
                metrics["style"] = "balanced"
            else:
                metrics["style"] = "other"

        row = {
            "run": run_num,
            "elapsed": elapsed,
            "tokens": resp.usage.total_tokens if resp.usage else 0,
            "has_content": bool(raw_text),
            "finish_reason": choice.finish_reason,
            "is_valid_lilypond": is_valid,
            "validation_error": validation_error,
            "raw_text": raw_text,
            "lilypond_code": lilypond_code
        }
        row.update(metrics)
        results.append(row)

        if not raw_text:
            display(Markdown(
                f"**⚠️ No content returned.**\n\n"
                f"- Finish reason: `{choice.finish_reason}`\n"
                f"- Time: {elapsed:.2f}s\n"
            ))
        else:
            output_md = f"### 🎵 Model Output\n```lilypond\n{lilypond_code}\n```\n\n"
            output_md += f"⏱️ Time: {elapsed:.2f}s | Tokens: {resp.usage.total_tokens if resp.usage else '—'}"
            if is_valid is not None:
                if is_valid:
                    output_md += " | ✅ **Compiles**"
                else:
                    output_md += f" | ❌ **Compile error**: {validation_error}"
            elif lilypond_code and not lilypond_path:
                output_md += " | ⚠️ *Not compiled (LilyPond not found)*"
            if metrics:
                failed = [k for k in (
                    "has_music_block","has_key_time","bars_ok","notes_per_bar_ok",
                    "durations_ok","lowercase_ok","no_forbidden","range_ok","notation_ok"
                ) if metrics.get(k) is False]
                output_md += (
                    f"\n\n**Constraints:** "
                    f"{int(metrics.get('adherence',0)*100)}% adherence • "
                    f"bars_ok={metrics.get('bars_ok')} • "
                    f"notes/bar_ok={metrics.get('notes_per_bar_ok')} • "
                    f"range_ok={metrics.get('range_ok')}"
                    f"\n**Diversity:** "
                    f"H={metrics.get('interval_entropy'):.2f} • "
                    f"repeat={metrics.get('repeat_rate'):.2f} • "
                    f"step%={(metrics.get('step_vs_leap') or 0.0)*100:.1f}% • "
                    f"dirchg={metrics.get('direction_changes')} • "
                    f"in-key={int(metrics.get('key_conform_pct',0)*100)}% • "
                    f"tonic-end={metrics.get('ends_on_tonic')}"
                    + (f"\n**Failed flags:** {', '.join(failed)}" if failed else "")
                )
                output_md += (
                    f"\n**Style:** {metrics.get('style','–')} • gates: "
                    f"E:{'✓' if metrics.get('entropy_ok') else '✗'} "
                    f"C:{'✓' if metrics.get('contour_ok') else '✗'} "
                    f"V:{'✓' if metrics.get('variety_ok') else '✗'}"
                    f"\n**Relative anchor:** "
                    f"c'={'✓' if metrics.get('relative_anchor_cprime') else '✗'} "
                    f"(plain c={'✓' if metrics.get('relative_anchor_plain_c') else '✗'})"
                )
            display(Markdown(output_md))
        display(Markdown("---\n"))

    # ===== Summary =====
    total_elapsed = (datetime.now() - total_start_time).total_seconds()
    N = len(results)
    successful_runs = sum(1 for r in results if r["has_content"])
    valid_lilypond_runs = sum(1 for r in results if r.get("is_valid_lilypond") == True)
    avg_time = sum(r["elapsed"] for r in results)/N if N else 0.0
    total_tokens = sum(r["tokens"] for r in results)

    def mean_of(field, filt=lambda r: True):
        vals = [r[field] for r in results if filt(r) and r.get(field) is not None]
        return (sum(vals)/len(vals)) if vals else 0.0
    def std_of(field, filt=lambda r: True):
        vals = [r[field] for r in results if filt(r) and r.get(field) is not None]
        if len(vals) < 2: return 0.0
        m = sum(vals)/len(vals)
        return (sum((x-m)**2 for x in vals)/(len(vals)-1))**0.5
    def count_of(field, filt=lambda r: True):
        vals = [r[field] for r in results if filt(r) and r.get(field) is not None]
        return len(vals)

    def strict_pass(r):
        needed = [
            "has_music_block",
            "bars_ok","notes_per_bar_ok","durations_ok","lowercase_ok",
            "no_forbidden","has_key_time","range_ok","notation_ok"
        ]
        return all(bool(r.get(k)) for k in needed)

    all_constraints_pass = sum(1 for r in results if strict_pass(r))

    adherence_mean = mean_of("adherence")
    H_mean, H_std = mean_of("interval_entropy"), std_of("interval_entropy")
    rep_mean, rep_std = mean_of("repeat_rate"), std_of("repeat_rate")
    k_mean, k_std = mean_of("key_conform_pct"), std_of("key_conform_pct")
    sl_mean, sl_std = mean_of("step_vs_leap"), std_of("step_vs_leap")
    dirchg_mean, dirchg_std = mean_of("direction_changes"), std_of("direction_changes")

    summary_md = f"""## 📊 Summary ({N} runs)

- **Responses with content:** {successful_runs}/{N} ({successful_runs/N*100:.1f}%)
{"- **Valid LilyPond code:** " + f"{valid_lilypond_runs}/{N} ({valid_lilypond_runs/N*100:.1f}%)" if lilypond_path else "- *(Compile check skipped — LilyPond not found)*"}

- **All-constraints pass (strict):** {all_constraints_pass}/{N} ({all_constraints_pass/N*100:.1f}%)
- **Mean adherence:** {adherence_mean:.3f}

**Runtime**
- Total time: {total_elapsed:.2f}s
- Avg time/run: {avg_time:.2f}s
- Total tokens: {total_tokens}
- Avg tokens/run: {total_tokens // max(1,N)}

### 🎼 Diversity (mean ± std)
- Interval entropy: {H_mean:.2f} ± {H_std:.2f} (n={count_of('interval_entropy')})
- Repeat rate: {rep_mean:.2f} ± {rep_std:.2f} (n={count_of('repeat_rate')})
- Step share: {sl_mean*100:.1f}% ± {sl_std*100:.1f}% (n={count_of('step_vs_leap')})
- Direction changes: {dirchg_mean:.2f} ± {dirchg_std:.2f} (n={count_of('direction_changes')})
- In-key %: {int(k_mean*100)}% ± {k_std*100:.2f}% (n={count_of('key_conform_pct')})
"""
    display(Markdown(summary_md))

    # --- Style distribution summary ---
    scalar_pct   = 100 * sum(1 for r in results if r.get("style") == "scalar")   / max(1, N)
    balanced_pct = 100 * sum(1 for r in results if r.get("style") == "balanced") / max(1, N)
    display(Markdown(f"**Styles:** scalar {scalar_pct:.0f}% • balanced {balanced_pct:.0f}%"))

    return results

# Example usage - call the function with default parameters
# results = run_zero_shot_test()
# Or with custom parameters:
# results = run_zero_shot_test(
#     prompt_file="../configs/prompts/zero_shot.txt",
#     model="openai/gpt-oss-20b:fireworks-ai",
#     num_runs=10,
#     max_tokens=10000,
#     temperature=0.7
# )

--------------------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Relative vs Absolute</h3>
</div>

## 🧪 Relative

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [30]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 6.35s | Tokens: 1029 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 |
  e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}
```

⏱️ Time: 7.38s | Tokens: 1014 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 20/20 (100.0%)
- **Mean adherence:** 1.000

**Runtime**
- Total time: 181.96s
- Avg time/run: 8.10s
- Total tokens: 21357
- Avg tokens/run: 1067

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.87 ± 0.00 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 100.0% ± 0.0% (n=20)
- Direction changes: 0.00 ± 0.00 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

--------------

## 🧪 Absolute

Prompt:
Write LilyPond code only.
Use \version "2.24.4" on the first line.
Then write exactly one music block enclosed in { ... }.
Inside that block, include \clef treble, \key c \major, \time 4/4, and \tempo 4 = 100 at the start.
Use only lowercase notes a–g with explicit durations (4).
Every note must include an octave mark relative to middle C (for example c' d' e' f').
Keep all pitches within the range c'..c''.
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout.

In [29]:
results = run_zero_shot_test(expected_notation="absolute")

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Use \version "2.24.4" on the first line.
Then write exactly one music block enclosed in { ... }.
Inside that block, include \clef treble, \key c \major, \time 4/4, and \tempo 4 = 100 at the start.
Use only lowercase notes a–g with explicit durations (4).
Every note must include an octave mark relative to middle C (for example c' d' e' f').
Keep all pitches within the range c'..c''.
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout.
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
{
  \clef treble
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4
}
```

⏱️ Time: 9.66s | Tokens: 1496 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=1.24 • repeat=0.00 • step%=90.3% • dirchg=6 • in-key=100% • tonic-end=1
**Style:** balanced • gates: E:✓ C:✓ V:✓
**Relative anchor:** c'=✗ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
{ \clef treble \key c \major \time 4/4 \tempo 4 = 100 c'4 d'4 e'4 f'4 | g'4 a'4 b'4 c''4 | d'4 e'4 f'4 g'4 | a'4 b'4 c'4 d'4 | e'4 f'4 g'4 a'4 | b'4 c''4 d'4 e'4 | f'4 g'4 a'4 b'4 | c''4 d'4 e'4 f'4 }
```

⏱️ Time: 11.09s | Tokens: 1774 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=1.42 • repeat=0.00 • step%=87.1% • dirchg=8 • in-key=100% • tonic-end=0
**Style:** balanced • gates: E:✓ C:✓ V:✓
**Relative anchor:** c'=✗ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 20/20 (100.0%)
- **Mean adherence:** 1.000

**Runtime**
- Total time: 247.43s
- Avg time/run: 11.34s
- Total tokens: 29277
- Avg tokens/run: 1463

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.81 ± 0.58 (n=20)
- Repeat rate: 0.06 ± 0.09 (n=20)
- Step share: 90.6% ± 10.9% (n=20)
- Direction changes: 4.65 ± 4.32 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 40% • balanced 60%

### Observation
Both syntaxes produced fully valid LilyPond code, but the relative syntax proved more efficient and consistent, generating shorter outputs and faster responses with uniform scalar motion. The absolute syntax showed greater melodic variation but at the cost of longer outputs and higher variability. For this project’s focus on reliability and reproducibility rather than creativity, the relative syntax was chosen as the canonical format due to its stability and efficiency.

----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Single-container vs Multi-container</h3>
</div>

## 🧪 Single Container

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other music at the top level).
At the start of that block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Write exactly 8 bars, each bar 4 quarter-notes.
Use only lowercase notes a–g with explicit 4 durations.
No rests, no chords, no repeats, no text, no \score or \layout, no << >> or \\.

In [27]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other music at the top level).
At the start of that block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Write exactly 8 bars, each bar 4 quarter-notes.
Use only lowercase notes a–g with explicit 4 durations.
No rests, no chords, no repeats, no text, no \score or \layout, no << >> or \\.
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4
}
```

⏱️ Time: 6.87s | Tokens: 1020 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 7.19s | Tokens: 878 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 14/20 (70.0%)
- **Mean adherence:** 0.933

**Runtime**
- Total time: 202.58s
- Avg time/run: 9.16s
- Total tokens: 25752
- Avg tokens/run: 1287

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.87 ± 0.00 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 100.0% ± 0.0% (n=20)
- Direction changes: 0.00 ± 0.00 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

----------------

## 🧪 Multi-container

Prompt:
Write LilyPond code only.
Use \version "2.24.4" on the first line.
Then write exactly two consecutive \relative c' { ... } blocks (no other music at the top level).
Inside the start of each block, include these three commands in this order:
\key c \major, \time 4/4, and \tempo 4 = 100.
Each block must have 4 bars, each bar containing 4 quarter-notes.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no repeats, no text, no \score, no \layout, no << >> or \\.

In [26]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Use \version "2.24.4" on the first line.
Then write exactly two consecutive \relative c' { ... } blocks (no other music at the top level).
Inside the start of each block, include these three commands in this order:
\key c \major, \time 4/4, and \tempo 4 = 100.
Each block must have 4 bars, each bar containing 4 quarter-notes.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no repeats, no text, no \score, no \layout, no << >> or \\.
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4
}
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4
}
```

⏱️ Time: 5.41s | Tokens: 872 | ✅ **Compiles**

**Constraints:** 66% adherence • bars_ok=False • notes/bar_ok=False • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** bars_ok, notes_per_bar_ok, notation_ok
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4
}
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4
}
```

⏱️ Time: 3.79s | Tokens: 694 | ✅ **Compiles**

**Constraints:** 66% adherence • bars_ok=False • notes/bar_ok=False • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** bars_ok, notes_per_bar_ok, notation_ok
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 0/20 (0.0%)
- **Mean adherence:** 0.678

**Runtime**
- Total time: 182.05s
- Avg time/run: 8.04s
- Total tokens: 19716
- Avg tokens/run: 985

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.02 ± 0.35 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 96.9% ± 10.7% (n=20)
- Direction changes: 0.35 ± 0.75 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 95% • balanced 5%

### Observation
Both configurations produced valid LilyPond syntax, but the single-container structure achieved much higher constraint adherence (70% vs 0%) and overall reliability. The multi-container setup led to systematic structural violations, suggesting the model struggles to maintain consistency across multiple independent \relative blocks. Although multi-container outputs were slightly shorter and marginally more varied, this came at the expense of correctness.
Therefore, the single-container format was adopted as the canonical structure for all subsequent experiments due to its superior stability, completeness, and alignment with evaluation goals.

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Inline variables/macros ON vs OFF</h3>
</div>

## 🧪 Inline variables/macros ON

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
Inline all variable or macro definitions directly inside the block, do not reference or define any external variables (no \foo, no custom macros, no # Scheme code).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [33]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
Inline all variable or macro definitions directly inside the block, do not reference or define any external variables (no \foo, no custom macros, no # Scheme code).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"

\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 7.73s | Tokens: 1090 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 5.58s | Tokens: 928 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 18/20 (90.0%)
- **Mean adherence:** 0.983

**Runtime**
- Total time: 164.32s
- Avg time/run: 7.22s
- Total tokens: 19228
- Avg tokens/run: 961

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.89 ± 0.07 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 99.2% ± 2.9% (n=20)
- Direction changes: 0.50 ± 1.82 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

---------------

## 🧪 Inline variables/macros OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then define one short musical variable (for example, melody = { c4 d4 e4 f4 | g4 a4 b4 c4 | ... }).
Inside that variable, include exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
Use only lowercase notes a–g with explicit durations (4).
After defining the variable, write one \relative c' { ... } block that contains:
\key c \major, \time 4/4, \tempo 4 = 100, and then the variable name (e.g. \melody).
Do not inline the note contents directly, use only the variable reference.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [34]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then define one short musical variable (for example, melody = { c4 d4 e4 f4 | g4 a4 b4 c4 | ... }).
Inside that variable, include exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
Use only lowercase notes a–g with explicit durations (4).
After defining the variable, write one \relative c' { ... } block that contains:
\key c \major, \time 4/4, \tempo 4 = 100, and then the variable name (e.g. \melody).
Do not inline the note contents directly, use only the variable reference.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
melody = { c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 }
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \melody
}
```

⏱️ Time: 5.28s | Tokens: 779 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"

\melody = {
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 |
  e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 |
}

\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \melody
}
```

⏱️ Time: 5.95s | Tokens: 823 | ❌ **Compile error**: test.ly:3:1: error: unknown escaped string: `\melody'

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 14/20 (70.0%)

- **All-constraints pass (strict):** 20/20 (100.0%)
- **Mean adherence:** 1.000

**Runtime**
- Total time: 199.46s
- Avg time/run: 8.94s
- Total tokens: 20647
- Avg tokens/run: 1032

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.87 ± 0.00 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 100.0% ± 0.0% (n=20)
- Direction changes: 0.00 ± 0.00 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

### Observation
Inlining variables/macros improved reliability and efficiency: it yielded 100% valid code, higher overall constraint satisfaction, faster runtime, and fewer tokens. Using external variables/macros (OFF) achieved perfect constraint adherence only on the runs that compiled, but suffered a 30% drop in syntactic validity, likely from name/scope/reference mistakes. Given our emphasis on robustness and reproducible evaluation, we adopt Inline ON as the canonical setting for subsequent experiments.

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Expanded repeats ON vs OFF</h3>
</div>

## 🧪 Expanded repeats ON

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Expand all repetitions manually — do not use \repeat, \repeat unfold, percent/measure repeats, tremolo shorthand, or any macros/variables for repetition. Write every note explicitly.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [35]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Expand all repetitions manually — do not use \repeat, \repeat unfold, percent/measure repeats, tremolo shorthand, or any macros/variables for repetition. Write every note explicitly.
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c'4 | d'4 e'4 f'4 g'4 | a'4 b'4 c''4 d''4 | e''4 f''4 g''4 a''4 | b''4 c'''4 d'''4 e'''4 | f'''4 g'''4 a'''4 b'''4 | c'''4 d'''4 e'''4 f'''4
}
```

⏱️ Time: 9.20s | Tokens: 1501 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=True • notes/bar_ok=True • range_ok=False
**Diversity:** H=1.09 • repeat=0.00 • step%=93.5% • dirchg=4 • in-key=100% • tonic-end=0
**Failed flags:** range_ok, notation_ok
**Style:** scalar • gates: E:✗ C:✓ V:✓
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 13.94s | Tokens: 972 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 19/20 (95.0%)
- **Mean adherence:** 0.989

**Runtime**
- Total time: 206.69s
- Avg time/run: 9.36s
- Total tokens: 21188
- Avg tokens/run: 1059

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.88 ± 0.05 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 99.7% ± 1.4% (n=20)
- Direction changes: 0.20 ± 0.89 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

--------------

## 🧪 Expanded repeats OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Inside the block, use exactly one \repeat unfold command to repeat a short melodic pattern until the total is 8 bars.
Do not expand repeats manually.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no text, no \score or \layout, and no voices (<< >> or \\).

In [36]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Inside the block, use exactly one \repeat unfold command to repeat a short melodic pattern until the total is 8 bars.
Do not expand repeats manually.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \repeat unfold 8 {
    c4 d4 e4 f4
  }
}
```

⏱️ Time: 6.36s | Tokens: 941 | ✅ **Compiles**

**Constraints:** 66% adherence • bars_ok=False • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.92 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** bars_ok, durations_ok, no_forbidden
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \repeat unfold 2 {
    c4 d4 e4 f4 g4 a4 b4 c4 d4 e4 f4 g4 a4 b4 c4 d4
  }
}
```

⏱️ Time: 6.44s | Tokens: 1148 | ✅ **Compiles**

**Constraints:** 55% adherence • bars_ok=False • notes/bar_ok=False • range_ok=True
**Diversity:** H=0.84 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** bars_ok, notes_per_bar_ok, durations_ok, no_forbidden
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 0/20 (0.0%)
- **Mean adherence:** 0.667

**Runtime**
- Total time: 162.28s
- Avg time/run: 7.11s
- Total tokens: 18912
- Avg tokens/run: 945

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.89 ± 0.03 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 100.0% ± 0.0% (n=20)
- Direction changes: 0.00 ± 0.00 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

### Observation
Expanding repeats ON delivered high reliability, 95% strict constraint adherence with 100% valid code, while the OFF setting (using a single \repeat unfold) was faster and shorter but failed all strict constraints. Musical diversity metrics were essentially identical. Given our emphasis on correctness and reproducibility, we adopt Expanded Repeats ON as the canonical setting.

----------------


<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Resolve or keep \transpose</h3>
</div>

## 🧪 Resolve \transpose ON

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Write all notes directly as if any transposition has already been applied. Do not use \transpose (or any macro/variable that performs transposition).
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [37]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Write all notes directly as if any transposition has already been applied. Do not use \transpose (or any macro/variable that performs transposition).
Use only lowercase notes a–g with explicit durations (4).
Write exactly 8 bars, each bar containing 4 quarter-notes, separated by |.
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 6.16s | Tokens: 1102 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 |
  e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
}
```

⏱️ Time: 4.96s | Tokens: 833 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 19/20 (95.0%)

- **All-constraints pass (strict):** 17/20 (85.0%)
- **Mean adherence:** 0.967

**Runtime**
- Total time: 163.60s
- Avg time/run: 7.16s
- Total tokens: 21601
- Avg tokens/run: 1080

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.88 ± 0.05 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 99.7% ± 1.0% (n=20)
- Direction changes: 0.20 ± 0.62 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

-----------------

## 🧪 Keep \transpose

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Inside the block, use exactly one \transpose FROM TO { ... } command that contains the entire 8-bar pattern. Do not expand the transposed notes manually.
The resulting output after transposition must total 8 bars, each bar 4 quarter-notes, separated by |.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).

In [39]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 100.
Inside the block, use exactly one \transpose FROM TO { ... } command that contains the entire 8-bar pattern. Do not expand the transposed notes manually.
The resulting output after transposition must total 8 bars, each bar 4 quarter-notes, separated by |.
Use only lowercase notes a–g with explicit durations (4).
No rests, no chords, no repeats, no text, no \score or \layout, and no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \transpose c c {
    a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4
  }
}
```

⏱️ Time: 8.65s | Tokens: 1502 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 100
  \transpose c d {
    c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
  }
}
```

⏱️ Time: 50.44s | Tokens: 2632 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 16/20 (80.0%)
- **Mean adherence:** 0.961

**Runtime**
- Total time: 273.70s
- Avg time/run: 12.66s
- Total tokens: 39031
- Avg tokens/run: 1951

### 🎼 Diversity (mean ± std)
- Interval entropy: 0.94 ± 0.18 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 99.0% ± 3.0% (n=20)
- Direction changes: 0.70 ± 1.87 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 100% • balanced 0%

### Observation
Both settings produced reliable outputs, but resolving \transpose offered better constraint adherence (85% vs 80%), lower latency, and far fewer tokens, simplifying normalization and evaluation. Keeping \transpose achieved slightly higher compile validity (100% vs 95%) and more melodic variety, but at substantial cost in runtime and verbosity. Given our emphasis on efficiency, reproducibility, and easy downstream parsing, we adopt Resolve \transpose ON as the canonical setting.

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 \tuplet vs \times</h3>
</div>

## 🧪 Normalize to \tuplet a/b { … }

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Write exactly 8 bars, separated by |.
In every bar, write exactly two \tuplet 3/2 { … } groups; each tuplet contains three eighth notes (8).
Fill the remaining two beats of each bar with two quarter notes (4).
Inside tuplets use eighths (8) only; outside tuplets use quarters (4) only.
Use only lowercase notes a–g with explicit durations.
No rests, no chords, no repeats, no text, no \score or \layout, no \transpose, no variables, no voices (<< >> or \\), no ties or slurs.

In [40]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Write exactly 8 bars, separated by |.
In every bar, write exactly two \tuplet 3/2 { … } groups; each tuplet contains three eighth notes (8).
Fill the remaining two beats of each bar with two quarter notes (4).
Inside tuplets use eighths (8) only; outside tuplets use quarters (4) only.
Use only lowercase notes a–g with explicit durations.
No rests, no chords, no repeats, no text, no \score or \layout, no \transpose, no variables, no voices (<< >> or \\), no ties or slurs.
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \tuplet 3/2 { c8 d8 e8 } \tuplet 3/2 { f8 g8 a8 } b4 c4 |
  \tuplet 3/2 { d8 e8 f8 } \tuplet 3/2 { g8 a8 b8 } c4 d4 |
  \tuplet 3/2 { e8 f8 g8 } \tuplet 3/2 { a8 b8 c8 } d4 e4 |
  \tuplet 3/2 { f8 g8 a8 } \tuplet 3/2 { b8 c8 d8 } e4 f4 |
  \tuplet 3/2 { g8 a8 b8 } \tuplet 3/2 { c8 d8 e8 } f4 g4 |
  \tuplet 3/2 { a8 b8 c8 } \tuplet 3/2 { d8 e8 f8 } g4 a4 |
  \tuplet 3/2 { b8 c8 d8 } \tuplet 3/2 { e8 f8 g8 } a4 b4 |
  \tuplet 3/2 { c8 d8 e8 } \tuplet 3/2 { f8 g8 a8 } b4 c4
}
```

⏱️ Time: 14.66s | Tokens: 2507 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=True • notes/bar_ok=False • range_ok=True
**Diversity:** H=1.51 • repeat=0.47 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=1
**Failed flags:** notes_per_bar_ok, durations_ok
**Style:** scalar • gates: E:✓ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \tuplet 3/2 { a8 b8 c8 } \tuplet 3/2 { d8 e8 f8 } g4 a4 |
  \tuplet 3/2 { b8 c8 d8 } \tuplet 3/2 { e8 f8 g8 } a4 b4 |
  \tuplet 3/2 { c8 d8 e8 } \tuplet 3/2 { f8 g8 a8 } b4 c4 |
  \tuplet 3/2 { d8 e8 f8 } \tuplet 3/2 { g8 a8 b8 } c4 d4 |
  \tuplet 3/2 { e8 f8 g8 } \tuplet 3/2 { a8 b8 c8 } d4 e4 |
  \tuplet 3/2 { f8 g8 a8 } \tuplet 3/2 { b8 c8 d8 } e4 f4 |
  \tuplet 3/2 { g8 a8 b8 } \tuplet 3/2 { c8 d8 e8 } f4 g4 |
  \tuplet 3/2 { a8 b8 c8 } \tuplet 3/2 { d8 e8 f8 } g4 a4
}
```

⏱️ Time: 15.10s | Tokens: 2674 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=True • notes/bar_ok=False • range_ok=True
**Diversity:** H=1.43 • repeat=0.47 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** notes_per_bar_ok, durations_ok
**Style:** scalar • gates: E:✓ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 0/20 (0.0%)
- **Mean adherence:** 0.778

**Runtime**
- Total time: 345.24s
- Avg time/run: 16.21s
- Total tokens: 56158
- Avg tokens/run: 2807

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.61 ± 0.25 (n=20)
- Repeat rate: 0.35 ± 0.19 (n=20)
- Step share: 90.3% ± 18.4% (n=20)
- Direction changes: 2.80 ± 5.37 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 75% • balanced 25%

------------------------

## 🧪 Keep \times a/b { … }

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Write exactly 8 bars, separated by |.
In every bar, write exactly two \times 2/3 { … } groups; each group contains three eighth notes (8).
Fill the remaining two beats of each bar with two quarter notes (4).
Inside tuplets use eighths (8) only; outside tuplets use quarters (4) only.
Use only lowercase notes a–g with explicit durations.
No rests, no chords, no repeats, no text, no \score or \layout, no \transpose, no variables, no voices (<< >> or \\), no ties or slurs.


In [41]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { ... } block containing all music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Write exactly 8 bars, separated by |.
In every bar, write exactly two \times 2/3 { … } groups; each group contains three eighth notes (8).
Fill the remaining two beats of each bar with two quarter notes (4).
Inside tuplets use eighths (8) only; outside tuplets use quarters (4) only.
Use only lowercase notes a–g with explicit durations.
No rests, no chords, no repeats, no text, no \score or \layout, no \transpose, no variables, no voices (<< >> or \\), no ties or slurs.
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \times 2/3 { c8 d8 e8 } \times 2/3 { f8 g8 a8 } c4 d4 |
  \times 2/3 { b8 c8 d8 } \times 2/3 { e8 f8 g8 } e4 f4 |
  \times 2/3 { a8 b8 c8 } \times 2/3 { d8 e8 f8 } g4 a4 |
  \times 2/3 { b8 c8 d8 } \times 2/3 { e8 f8 g8 } b4 c4 |
  \times 2/3 { c8 d8 e8 } \times 2/3 { f8 g8 a8 } c4 d4 |
  \times 2/3 { b8 c8 d8 } \times 2/3 { e8 f8 g8 } e4 f4 |
  \times 2/3 { a8 b8 c8 } \times 2/3 { d8 e8 f8 } g4 a4 |
  \times 2/3 { b8 c8 d8 } \times 2/3 { e8 f8 g8 } b4 c4
}
```

⏱️ Time: 12.88s | Tokens: 2379 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=True • notes/bar_ok=False • range_ok=True
**Diversity:** H=1.16 • repeat=0.07 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=1
**Failed flags:** notes_per_bar_ok, durations_ok
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \times 2/3 { a8 b8 c8 } \times 2/3 { d8 e8 f8 } g4 a4 |
  \times 2/3 { b8 c8 d8 } \times 2/3 { e8 f8 g8 } a4 b4 |
  \times 2/3 { c8 d8 e8 } \times 2/3 { f8 g8 a8 } b4 c4 |
  \times 2/3 { d8 e8 f8 } \times 2/3 { g8 a8 b8 } c4 d4 |
  \times 2/3 { e8 f8 g8 } \times 2/3 { a8 b8 c8 } d4 e4 |
  \times 2/3 { f8 g8 a8 } \times 2/3 { b8 c8 d8 } e4 f4 |
  \times 2/3 { g8 a8 b8 } \times 2/3 { c8 d8 e8 } f4 g4 |
  \times 2/3 { a8 b8 c8 } \times 2/3 { d8 e8 f8 } g4 a4
}
```

⏱️ Time: 19.52s | Tokens: 3527 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=True • notes/bar_ok=False • range_ok=True
**Diversity:** H=1.43 • repeat=0.47 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Failed flags:** notes_per_bar_ok, durations_ok
**Style:** scalar • gates: E:✓ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 0/20 (0.0%)
- **Mean adherence:** 0.778

**Runtime**
- Total time: 321.87s
- Avg time/run: 15.05s
- Total tokens: 51921
- Avg tokens/run: 2596

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.51 ± 0.29 (n=20)
- Repeat rate: 0.24 ± 0.22 (n=20)
- Step share: 88.7% ± 17.6% (n=20)
- Direction changes: 3.50 ± 5.61 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 60% • balanced 30%

### Observation
Both tupleting formulations (\tuplet 3/2 and \times 2/3) yielded 100% valid syntax but 0% strict constraint pass, indicating the model systematically mis-handled the tuplet–meter arithmetic despite clear instructions. Performance and length slightly favored \times (≈7% faster, ≈8% fewer tokens), while musical diversity differences were small. For benchmarking we therefore prefer \times as the canonical form on efficiency grounds, but note that tuplets remain a failure mode; future prompts should include a worked bar example or relax constraints if tuplets are required.

-----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">🎵 Expand music functions ON vs OFF</h3>
</div>

## 🧪 Expand music functions ON

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { … } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Use only lowercase notes a–g with explicit durations 4 or 8 (no other durations).
Write exactly 8 bars, each bar totaling 4 quarter-notes; bars are separated by |. You may mix quarters and eighths, but no tuplets, no dotted notes, and no ties or slurs.
Do not define or use any music functions (no #(define-music-function …), no #(define …), no Scheme), and no variables/macros (no \foo).
No rests, no chords, no repeats, no \transpose, no \score or \layout, no text, no voices (<< >> or \\).

In [42]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one \relative c' { … } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Use only lowercase notes a–g with explicit durations 4 or 8 (no other durations).
Write exactly 8 bars, each bar totaling 4 quarter-notes; bars are separated by |. You may mix quarters and eighths, but no tuplets, no dotted notes, and no ties or slurs.
Do not define or use any music functions (no #(define-music-function …), no #(define …), no Scheme), and no variables/macros (no \foo).
No rests, no chords, no repeats, no \transpose, no \score or \layout, no text, no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  c4 d4 e4 f4 | g8 a8 b8 c8 | d4 e4 f4 g4 | a8 b8 c8 d8 | e4 f4 g4 a4 | b8 c8 d8 e8 | f4 g4 a4 b4 | c8 d8 e8 f8
}
```

⏱️ Time: 6.15s | Tokens: 1087 | ✅ **Compiles**

**Constraints:** 77% adherence • bars_ok=False • notes/bar_ok=True • range_ok=True
**Diversity:** H=1.55 • repeat=0.00 • step%=80.0% • dirchg=6 • in-key=100% • tonic-end=0
**Failed flags:** bars_ok, durations_ok
**Style:** balanced • gates: E:✓ C:✓ V:✓
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"
\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  c4 d4 e4 f4 | a4 b4 c8 d8 e8 f8 | g8 a8 b8 c8 d8 e8 f8 g8 | a4 b4 c4 d4 | e4 f4 g8 a8 b8 c8 | d8 e8 f8 g8 a8 b8 c8 d8 | e4 f4 g4 a4 | b4 c4 d8 e8 f8 g8
}
```

⏱️ Time: 20.58s | Tokens: 3499 | ✅ **Compiles**

**Constraints:** 66% adherence • bars_ok=False • notes/bar_ok=False • range_ok=True
**Diversity:** H=1.73 • repeat=0.00 • step%=94.1% • dirchg=4 • in-key=100% • tonic-end=1
**Failed flags:** bars_ok, notes_per_bar_ok, durations_ok
**Style:** balanced • gates: E:✓ C:✓ V:✓
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 20/20 (100.0%)

- **All-constraints pass (strict):** 9/20 (45.0%)
- **Mean adherence:** 0.856

**Runtime**
- Total time: 301.59s
- Avg time/run: 14.03s
- Total tokens: 39262
- Avg tokens/run: 1963

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.43 ± 0.50 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 88.6% ± 11.4% (n=20)
- Direction changes: 2.55 ± 3.90 (n=20)
- In-key %: 100% ± 0.00% (n=20)


**Styles:** scalar 40% • balanced 40%

----------

## 🧪 Expand music functions OFF

Prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Define exactly one identity music function named \id that returns its argument unchanged, using:

id = #(define-music-function (parser location music) (ly:music?)
  #{ $music #})


Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Inside the block, call \id exactly once and use it to wrap the entire melody (nothing musical outside the \id { ... } braces).
Inside \id { ... }, write exactly 8 bars separated by |, each bar totaling 4 quarter-note beats.
Use only lowercase notes a–g with explicit durations 4 or 8 (no other durations). No tuplets, no dotted notes, no ties or slurs.
No rests, no chords, no repeats, no \transpose, no \score or \layout, no other variables/macros or Scheme, no text, no voices (<< >> or \\).

In [43]:
results = run_zero_shot_test()

✅ **LilyPond found:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`


### 📝 Input Prompt
```
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Define exactly one identity music function named \id that returns its argument unchanged, using:

id = #(define-music-function (parser location music) (ly:music?)
  #{ $music #})


Then write exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of the block, include \key c \major, \time 4/4, and \tempo 4 = 120.
Inside the block, call \id exactly once and use it to wrap the entire melody (nothing musical outside the \id { ... } braces).
Inside \id { ... }, write exactly 8 bars separated by |, each bar totaling 4 quarter-note beats.
Use only lowercase notes a–g with explicit durations 4 or 8 (no other durations). No tuplets, no dotted notes, no ties or slurs.
No rests, no chords, no repeats, no \transpose, no \score or \layout, no other variables/macros or Scheme, no text, no voices (<< >> or \\).
```

---


## 🔄 Run 1/20


### 🎵 Model Output
```lilypond
\version "2.24.4"

id = #(define-music-function (parser location music) (ly:music?)
  #{ $music #})

\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \id {
    c4 d4 e4 f4 | g4 f4 e4 d4 | c4 e4 g4 b4 | a4 g4 f4 e4 |
    d4 c4 b4 a4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4
  }
}
```

⏱️ Time: 9.67s | Tokens: 1636 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=2.18 • repeat=0.00 • step%=90.3% • dirchg=4 • in-key=100% • tonic-end=0
**Style:** balanced • gates: E:✓ C:✓ V:✓
**Relative anchor:** c'=✓ (plain c=✗)

---


...

## 🔄 Run 20/20


### 🎵 Model Output
```lilypond
\version "2.24.4"

id = #(define-music-function (parser location music) (ly:music?)
  #{ $music #})

\relative c' {
  \key c \major
  \time 4/4
  \tempo 4 = 120
  \id {
    c4 d4 e4 f4 | g4 a4 b4 c4 | d4 e4 f4 g4 | a4 b4 c4 d4 | e4 f4 g4 a4 | b4 c4 d4 e4 | f4 g4 a4 b4 | c4 d4 e4 f4
  }
}
```

⏱️ Time: 25.69s | Tokens: 1586 | ✅ **Compiles**

**Constraints:** 100% adherence • bars_ok=True • notes/bar_ok=True • range_ok=True
**Diversity:** H=0.87 • repeat=0.00 • step%=100.0% • dirchg=0 • in-key=100% • tonic-end=0
**Style:** scalar • gates: E:✗ C:✗ V:✗
**Relative anchor:** c'=✓ (plain c=✗)

---


## 📊 Summary (20 runs)

- **Responses with content:** 20/20 (100.0%)
- **Valid LilyPond code:** 19/20 (95.0%)

- **All-constraints pass (strict):** 16/20 (80.0%)
- **Mean adherence:** 0.933

**Runtime**
- Total time: 265.58s
- Avg time/run: 12.26s
- Total tokens: 38458
- Avg tokens/run: 1922

### 🎼 Diversity (mean ± std)
- Interval entropy: 1.04 ± 0.51 (n=20)
- Repeat rate: 0.00 ± 0.00 (n=20)
- Step share: 90.1% ± 25.9% (n=20)
- Direction changes: 0.50 ± 1.28 (n=20)
- In-key %: 95% ± 22.36% (n=20)


**Styles:** scalar 85% • balanced 10%

### Observation
Allowing a minimal identity function (\id) increased constraint adherence (80% vs 45%) and slightly improved speed/conciseness, but at the cost of lower compile validity (95% vs 100%) and added normalization complexity due to embedded Scheme. Since our benchmarks prioritize robustness, portability, and simple preprocessing, we adopt Expand music functions ON (no functions) as the canonical setting and treat the \id wrapper as a non-canonical shortcut that can inflate adherence while complicating the pipeline.

## Conclusion

The prompt canonicalization phase successfully identified the optimal structural configuration for reliable LilyPond generation.
Through systematic A/B testing, we demonstrated that simplifying the prompt syntax, using a single \relative block, inlined content, expanded repetitions, resolved transpositions, and no Scheme or macro constructs, consistently yields the highest validity and adherence rates.
These findings establish a standardized baseline prompt template that will be used across all subsequent zero-shot and fine-tuning experiments to ensure reproducibility, comparability, and stable model behavior.

## Final Baseline LilyPond Prompt Template

Write LilyPond code only.
\version "2.24.4"

Use exactly one \relative c' { ... } block containing all the music (no other top-level music).
At the start of that block, include in this order:
\key c \major
\time 4/4
\tempo 4 = 120

Write exactly 8 bars, each bar containing 4 beats in total, separated by |.
Use only lowercase notes a–g with explicit durations (4 or 8).
You may mix quarter and eighth notes, but do not use tuplets, dotted rhythms, ties, slurs, or rests.

No chords, no repeats, no \transpose, no macros or variables, no Scheme code, no music functions, no text, and no voices (<< >> or \\).
Do not include \score, \layout, or any additional markup.

Ensure the code compiles as-is in LilyPond 2.24.4.
